# 이미지 생성 애플리케이션 제작하기 (OpenAI)

LLM은 단순한 텍스트 생성 이상의 기능을 갖추고 있습니다. 텍스트 설명으로부터 이미지를 생성할 수도 있습니다. 의료기술, 건축, 관광, 게임 개발, 마케팅 등 다양한 분야에서 이미지 모달리티가 유용합니다. 이 수업에서는 OpenAI 플랫폼의 최신 **GPT 이미지** 모델을 다룹니다.

## 학습 목표

이 수업이 끝나면 다음을 할 수 있습니다:

- 이미지 생성이 무엇인지, 어디에 유용한지 설명할 수 있습니다.
- `gpt-image` 모델 군을 이해하고, 기존 DALL·E 모델과 어떻게 다른지 알 수 있습니다.
- 이미지 생성 애플리케이션을 구축하고 이미지를 편집할 수 있습니다.

## 이미지 생성이란?

이미지 생성 모델은 텍스트 프롬프트로부터 이미지를 만듭니다. `gpt-image` 같은 최신 모델은 훈련 과정에서 텍스트와 이미지 간의 관계를 학습하고, 무작위 노이즈를 반복적으로 변환하여 프롬프트와 일치하는 이미지를 만들어냅니다.

잘 알려진 두 가지 이미지 모델 계열은 다음과 같습니다:

- **`gpt-image` (OpenAI)** — 이 수업에서 사용하는 최신 세대 모델로, 텍스트-투-이미지 생성과 이미지 편집(마스크를 이용한 인페인팅)을 지원합니다.
- **Midjourney** — 독자적인 서비스와 Discord 기반 워크플로를 가진 인기 있는 타사 모델입니다.

> 이전의 OpenAI 이미지 모델인 — **DALL·E 2** 와 **DALL·E 3** 는 레거시 모델입니다. DALL·E 3는 신규 배포에 더 이상 제공되지 않으며, `create_variation` 기능은 DALL·E 2에서만 존재했습니다. 새로운 애플리케이션에는 `gpt-image` 모델을 사용하세요.

> **중요:** `gpt-image` 모델은 생성된 이미지를 URL이 아닌 **base64** (`b64_json`) 형태로 반환합니다. 코드가 base64 문자열을 디코드하여 바이트로 저장하므로, 다운로드할 수 있는 이미지 URL은 없습니다.


## 첫 번째 이미지 생성 애플리케이션 구축하기

그럼 이미지 생성 애플리케이션을 만들기 위해 필요한 것은 무엇일까요? 다음 라이브러리들이 필요합니다:

- **python-dotenv**, 비밀 정보를 코드에서 분리된 *.env* 파일에 보관하기 위해 이 라이브러리를 사용하는 것이 매우 권장됩니다.
- **openai**, OpenAI API와 상호작용하기 위해 사용할 라이브러리입니다.
- **pillow**, Python에서 이미지를 다루기 위한 라이브러리입니다.


1. 다음 내용을 포함하는 *.env* 파일을 만듭니다:

    ```text
    OPENAI_API_KEY='<add your OpenAI key here>'
    ```



1. 위의 라이브러리들을 *requirements.txt* 파일에 다음과 같이 모으세요:

    ```text
    python-dotenv
    openai
    pillow
    ```


1. 다음으로, 가상 환경을 만들고 라이브러리를 설치하세요:


In [ ]:
# create virtual env
! python3 -m venv venv
# activate environment
! source venv/bin/activate
# install libraries
# pip install -r requirements.txt, if using a requirements.txt file 
! pip install python-dotenv openai pillow

> [!NOTE]
> Windows에서는 다음 명령어를 사용하여 가상 환경을 생성하고 활성화하세요:

    ```bash
    python3 -m venv venv
    venv\Scripts\activate.bat
    ````

1. <em>app.py</em>라는 파일에 다음 코드를 추가합니다:

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai

    # dotenv를 가져옵니다
    dotenv.load_dotenv()

    # OpenAI 객체 생성 (.env에서 OPENAI_API_KEY를 읽음)
    client = openai.OpenAI()


    try:
        # 이미지 생성 API를 사용하여 이미지 생성
        generation_response = client.images.generate(
            model="gpt-image-1",
            prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 여기에 프롬프트 텍스트 입력
            size='1024x1024',
            n=1
        )
        # 저장할 이미지 디렉토리 설정
        image_dir = os.path.join(os.curdir, 'images')

        # 디렉토리가 없으면 생성
        if not os.path.isdir(image_dir):
            os.mkdir(image_dir)

        # 이미지 경로 초기화 (파일 형식은 png여야 함)
        image_path = os.path.join(image_dir, 'generated-image.png')

        # gpt-image 모델은 URL이 아닌 base64(b64_json) 형식으로 이미지를 반환함
        image_b64 = generation_response.data[0].b64_json
        generated_image = base64.b64decode(image_b64)
        with open(image_path, "wb") as image_file:
            image_file.write(generated_image)

        # 기본 이미지 뷰어에서 이미지 표시
        image = Image.open(image_path)
        image.show()

    # 예외 처리
    except openai.BadRequestError as err:
        print(err)

    ```

이 코드를 설명해 보겠습니다:

- 먼저, OpenAI 라이브러리, dotenv 라이브러리, base64 모듈, 그리고 Pillow 라이브러리를 포함하여 필요한 라이브러리를 가져옵니다.

    ```python
    import os
    import base64
    from PIL import Image
    import dotenv
    import openai
    ```

- 그 후, 클라이언트를 생성하는데, 여기서 API 키를 ``.env``에서 읽어옵니다.

    ```python
    # OpenAI 객체 생성
    client = openai.OpenAI()
    ```

- 다음으로, 이미지를 생성합니다:

    ```python
    generation_response = client.images.generate(
        model="gpt-image-1",
        prompt='Bunny on horse, holding a lollipop, on a foggy meadow where it grows daffodils',    # 여기에 프롬프트 텍스트를 입력하세요
        size='1024x1024',
        n=1
    )
    ```

    `gpt-image` 모델은 이미지를 `data[0].b64_json`에 있는 **base64** 문자열로 반환합니다. 이를 바이트로 디코딩한 후 파일로 씁니다 — 다운로드할 URL은 없습니다.

    ```python
    image_b64 = generation_response.data[0].b64_json
    generated_image = base64.b64decode(image_b64)
    with open(image_path, "wb") as image_file:
        image_file.write(generated_image)
    ```

- 마지막으로, 이미지를 열고 표준 이미지 뷰어를 사용해 표시합니다:

    ```python
    image = Image.open(image_path)
    image.show()
    ```

### 이미지 생성에 대한 자세한 내용

`images.generate`의 매개변수를 살펴보겠습니다:

- <strong>model</strong>은 사용할 이미지 모델입니다 (예: `gpt-image-1`).
- <strong>prompt</strong>는 이미지를 생성하는 데 사용되는 텍스트 프롬프트입니다. 여기서는 "사탕을 들고 안개 낀 꽃밭에 서 있는 토끼"입니다.
- <strong>size</strong>는 생성할 이미지의 크기입니다 (예: `1024x1024`, `1536x1024`, `1024x1536` 또는 `"auto"`).
- <strong>n</strong>은 생성할 이미지 수입니다. 여기서는 한 장을 생성합니다.

> 이미지 모델은 `temperature` 매개변수를 사용하지 않습니다 — 이는 텍스트 생성 제어에 해당합니다. 다양성을 원하면 API를 다시 호출하고, 다양성을 줄이려면 프롬프트를 더 구체적으로 만드세요.

## 이미지 생성의 추가 기능

몇 줄의 Python 코드로 이미지를 생성하는 방법을 보셨습니다. `gpt-image` 모델은 기존 이미지를 <strong>편집</strong>할 수도 있습니다. 이미지, 선택적 <strong>마스크</strong>(변경할 영역 표시), 그리고 프롬프트를 제공하여 이미지의 일부를 변경할 수 있습니다 — 예를 들어 토끼에게 모자를 씌우는 것처럼요.

```python
response = client.images.edit(
  model="gpt-image-1",
  image=open("base_image.png", "rb"),
  mask=open("mask.png", "rb"),
  prompt="An image of a rabbit with a hat on its head.",
  n=1,
  size="1024x1024"
)
# 편집 내용도 base64로 반환됩니다
edited_image = base64.b64decode(response.data[0].b64_json)
```

기본 이미지는 토끼만 포함하고 있으며, 최종 이미지에는 토끼가 모자를 쓰고 있습니다.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**면책 조항**:
이 문서는 AI 번역 서비스 [Co-op Translator](https://github.com/Azure/co-op-translator)를 사용하여 번역되었습니다. 정확성을 기하기 위해 노력하고 있으나, 자동 번역은 오류나 부정확한 부분이 있을 수 있음을 유의하시기 바랍니다. 원본 문서의 원어본이 권위 있는 자료로 간주되어야 합니다. 중요한 정보의 경우, 전문가의 인간 번역을 권장합니다. 이 번역 사용으로 인해 발생하는 오해나 잘못된 해석에 대해 당사는 책임을 지지 않습니다.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
